In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\335594\OneDrive - NTT DATA, Inc\Desktop\Hackathon\dbiz


In [3]:
# configure logging
from src.observability.logging_config import configure_logging

configure_logging()

2026-09-19 23:05:47,546 | INFO | src.observability.logging_config | Logging configured | level=INFO


In [4]:
# verify imports
from src.ingestion import load_pdf_directory
from src.chunking import chunk_pages

from src.providers.embeddings.factory import get_embedding_provider
from src.providers.vector_store.factory import get_vector_store

print("All imports successful")

All imports successful


In [5]:
# ingest PDFs
documents = load_pdf_directory(
    "../data/pdfs"
)

print("Extracted pages:", len(documents))

2026-09-19 23:05:50,047 | INFO | src.ingestion | Starting corpus ingestion | directory=..\data\pdfs | pdf_count=20
2026-09-19 23:05:50,048 | INFO | src.ingestion | Extracting PDF | source=2022 Q3 AAPL.pdf
2026-09-19 23:05:50,183 | INFO | src.ingestion | PDF extraction completed | source=2022 Q3 AAPL.pdf | pages=28
2026-09-19 23:05:50,185 | INFO | src.ingestion | Extracting PDF | source=2022 Q3 AMZN.pdf
2026-09-19 23:05:50,368 | INFO | src.ingestion | PDF extraction completed | source=2022 Q3 AMZN.pdf | pages=50
2026-09-19 23:05:50,369 | INFO | src.ingestion | Extracting PDF | source=2022 Q3 INTC.pdf
2026-09-19 23:05:50,797 | INFO | src.ingestion | PDF extraction completed | source=2022 Q3 INTC.pdf | pages=61
2026-09-19 23:05:50,798 | INFO | src.ingestion | Extracting PDF | source=2022 Q3 MSFT.pdf
2026-09-19 23:05:52,509 | INFO | src.ingestion | PDF extraction completed | source=2022 Q3 MSFT.pdf | pages=80
2026-09-19 23:05:52,510 | INFO | src.ingestion | Extracting PDF | source=2022 Q3 

Extracted pages: 1035


In [6]:
# inspect one document:
documents[0]

{'text': 'UNITED STATES\nSECURITIES AND EXCHANGE COMMISSION\nWashington, D.C. 20549\nFORM 10-Q\n(Mark One)\n☒ QUARTERLY REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934\nFor the quarterly period ended June\xa025, 2022\nor\n☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934\nFor the transition period from \xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0 to \xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0.\nCommission File Number: 001-36743\nApple Inc.\n(Exact name of Registrant as specified in its charter)\nCalifornia\n94-2404110\n(State or other jurisdiction\nof incorporation or organization)\n(I.R.S. Employer Identification No.)\nOne Apple Park Way\nCupertino, California\n95014\n(Address of principal executive offices)\n(Zip Code)\n(408) 996-1010\n(Registrant’s telephone number, including area code)\nSecurities registered pursuant to Section 12(b) of the Act:\nTitle of each class\nTrading symbol(s)\nName of each exch

In [7]:
# chunk documents
chunks = chunk_pages(documents)

print("Pages:", len(documents))
print("Chunks:", len(chunks))

2026-09-19 23:06:00,087 | INFO | src.chunking | Starting chunking | page_count=1035
2026-09-19 23:06:01,722 | INFO | src.chunking | Chunking completed | chunks=1260 | filtered=2


Pages: 1035
Chunks: 1260


In [8]:
chunks[0].keys()

dict_keys(['chunk_id', 'text', 'source', 'page', 'file_path', 'chunk_index', 'token_count'])

In [9]:
# verify chunk statistics
import pandas as pd

chunk_df = pd.DataFrame([
    {
        "chunk_id": chunk["chunk_id"],
        "source": chunk["source"],
        "page": chunk["page"],
        "tokens": chunk["token_count"],
    }
    for chunk in chunks
])

chunk_df["tokens"].describe()

count    1260.000000
mean      609.084921
std       267.391245
min        55.000000
25%       382.750000
50%       652.500000
75%       849.250000
max       992.000000
Name: tokens, dtype: float64

In [23]:
chunk_df

,chunk_id,source,page,tokens
0,2022_Q3_AAPL_p1_c1,2022 Q3 AAPL.pdf,1,755
1,2022_Q3_AAPL_p2_c1,2022 Q3 AAPL.pdf,2,117
2,2022_Q3_AAPL_p3_c1,2022 Q3 AAPL.pdf,3,166
3,2022_Q3_AAPL_p4_c1,2022 Q3 AAPL.pdf,4,551
4,2022_Q3_AAPL_p5_c1,2022 Q3 AAPL.pdf,5,373
...,...,...,...,...
1255,2023_Q3_NVDA_p48_c1,2023 Q3 NVDA.pdf,48,100
1256,2023_Q3_NVDA_p49_c1,2023 Q3 NVDA.pdf,49,661
1257,2023_Q3_NVDA_p50_c1,2023 Q3 NVDA.pdf,50,660
1258,2023_Q3_NVDA_p51_c1,2023 Q3 NVDA.pdf,51,403


In [10]:
assert len(chunks) > 0
assert chunk_df["tokens"].min() >= 50
assert chunk_df["tokens"].max() <= 1000

In [11]:
# initialize embedding provider
embedding_provider = get_embedding_provider()

print(
    "Embedding dimension:",
    embedding_provider.dimension
)

2026-09-19 23:06:02,823 | INFO | src.providers.embeddings.factory | Creating embedding provider | provider=azure_openai
2026-09-19 23:06:02,894 | INFO | src.providers.embeddings.azure_openai | Azure embedding provider initialized | dimension=3072


Embedding dimension: 3072


In [12]:
# test one query embedding
query_vector = embedding_provider.embed_query(
    "What were Apple's total net sales?"
)

print("Vector length:", len(query_vector))
print("First 5 values:", query_vector[:5])

2026-09-19 23:06:04,062 | INFO | httpx2 | HTTP Request: POST https://qubinexa-dev-ai-resource.services.ai.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2026-09-19 23:06:04,066 | INFO | src.observability.tracing | Trace completed | step=azure_query_embedding | duration_seconds=1.126


Vector length: 3072
First 5 values: [-0.022125244140625, -0.01197052001953125, -0.01160430908203125, 0.029571533203125, -0.011383056640625]


In [13]:
assert len(query_vector) == embedding_provider.dimension

In [14]:
# test batch embeddings
sample_texts = [
    chunk["text"]
    for chunk in chunks[:5]
]

sample_vectors = embedding_provider.embed_texts(
    sample_texts
)

print("Vectors returned:", len(sample_vectors))
print("Vector dimension:", len(sample_vectors[0]))

2026-09-19 23:06:04,837 | INFO | httpx2 | HTTP Request: POST https://qubinexa-dev-ai-resource.services.ai.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2026-09-19 23:06:04,907 | INFO | src.observability.tracing | Trace completed | step=azure_embedding_batch | duration_seconds=0.754


Vectors returned: 5
Vector dimension: 3072


In [15]:
assert len(sample_vectors) == 5

assert all(
    len(vector) == embedding_provider.dimension
    for vector in sample_vectors
)

print("Embedding validation passed")

Embedding validation passed


In [16]:
# initialize Azure AI Search
vector_store = get_vector_store(
    embedding_dimension=embedding_provider.dimension
)

print(
    "Vector store:",
    type(vector_store).__name__
)

2026-09-19 23:06:05,007 | INFO | src.providers.vector_store.factory | Creating vector store | provider=azure_search
2026-09-19 23:06:05,012 | INFO | src.providers.vector_store.azure_search | Azure AI Search vector store initialized | index=rag-index | dimension=3072


Vector store: AzureAISearchVectorStore


In [17]:
# create the index
vector_store.create_index()

2026-09-19 23:06:05,067 | INFO | src.providers.vector_store.azure_search | Creating Azure AI Search index | index=rag-index
2026-09-19 23:06:06,274 | INFO | src.observability.tracing | Trace completed | step=azure_search_create_index | duration_seconds=1.201
2026-09-19 23:06:06,275 | INFO | src.providers.vector_store.azure_search | Azure AI Search index ready | index=rag-index


In [18]:
# prepare five embedded chunks
sample_chunks = chunks[:5]

sample_texts = [
    chunk["text"]
    for chunk in sample_chunks
]

sample_vectors = embedding_provider.embed_texts(
    sample_texts
)

2026-09-19 23:06:07,452 | INFO | httpx2 | HTTP Request: POST https://qubinexa-dev-ai-resource.services.ai.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2026-09-19 23:06:07,501 | INFO | src.observability.tracing | Trace completed | step=azure_embedding_batch | duration_seconds=1.178


In [19]:
sample_embedded_chunks = [
    {
        **chunk,
        "embedding": vector,
    }
    for chunk, vector in zip(
        sample_chunks,
        sample_vectors,
    )
]

In [20]:
sample_embedded_chunks[0].keys()

dict_keys(['chunk_id', 'text', 'source', 'page', 'file_path', 'chunk_index', 'token_count', 'embedding'])

In [21]:
len(
    sample_embedded_chunks[0]["embedding"]
)

3072

In [22]:
# upload five chunks
vector_store.add_documents(
    sample_embedded_chunks
)

2026-09-19 23:06:07,688 | INFO | src.providers.vector_store.azure_search | Uploading documents to Azure AI Search | count=5 | index=rag-index
2026-09-19 23:06:09,853 | INFO | src.observability.tracing | Trace completed | step=azure_search_document_upload | duration_seconds=2.164
2026-09-19 23:06:09,853 | INFO | src.providers.vector_store.azure_search | Azure AI Search upload completed | uploaded=5


In [24]:
# inspect the five uploaded chunks
for i, chunk in enumerate(
    sample_chunks,
    start=1,
):
    print("=" * 100)
    print("Chunk:", i)
    print("Source:", chunk["source"])
    print("Page:", chunk["page"])
    print()
    print(chunk["text"][:1000])

Chunk: 1
Source: 2022 Q3 AAPL.pdf
Page: 1

UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-Q
(Mark One)
☒ QUARTERLY REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the quarterly period ended June 25, 2022
or
☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the transition period from              to             .
Commission File Number: 001-36743
Apple Inc.
(Exact name of Registrant as specified in its charter)
California
94-2404110
(State or other jurisdiction
of incorporation or organization)
(I.R.S. Employer Identification No.)
One Apple Park Way
Cupertino, California
95014
(Address of principal executive offices)
(Zip Code)
(408) 996-1010
(Registrant’s telephone number, including area code)
Securities registered pursuant to Section 12(b) of the Act:
Title of each class
Trading symbol(s)
Name of each exchange on which registered
Common Stock, $0.00001 par value per share
AA

In [25]:
# vector retrieval
query = "What financial results are reported by Apple?"

query_vector = embedding_provider.embed_query(
    query
)

results = vector_store.search(
    query_vector=query_vector,
    top_k=3,
)

2026-09-19 23:08:26,708 | INFO | httpx2 | HTTP Request: POST https://qubinexa-dev-ai-resource.services.ai.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2026-09-19 23:08:26,710 | INFO | src.observability.tracing | Trace completed | step=azure_query_embedding | duration_seconds=0.978
2026-09-19 23:08:26,717 | INFO | src.providers.vector_store.azure_search | Executing vector search | index=rag-index | top_k=3
2026-09-19 23:08:28,776 | INFO | src.observability.tracing | Trace completed | step=azure_search_vector_query | duration_seconds=2.057
2026-09-19 23:08:28,777 | INFO | src.providers.vector_store.azure_search | Vector search completed | results=3


In [26]:
for rank, result in enumerate(
    results,
    start=1,
):
    print("=" * 100)
    print("Rank:", rank)
    print("Score:", result["score"])
    print("Source:", result["source"])
    print("Page:", result["page"])
    print("Chunk ID:", result["chunk_id"])
    print()
    print(result["text"][:1200])

Rank: 1
Score: 0.74589527
Source: 2022 Q3 AAPL.pdf
Page: 4
Chunk ID: 2022_Q3_AAPL_p4_c1

PART I — FINANCIAL INFORMATION
Item 1.    Financial Statements
Apple Inc.
CONDENSED CONSOLIDATED STATEMENTS OF OPERATIONS (Unaudited)
(In millions, except number of shares which are reflected in thousands and per share amounts)
Three Months Ended
Nine Months Ended
June 25,
2022
June 26,
2021
June 25,
2022
June 26,
2021
Net sales:
   Products
$
63,355 
$
63,948 
$
245,241 
$
232,309 
   Services
19,604 
17,486 
58,941 
50,148 
Total net sales
82,959 
81,434 
304,182 
282,457 
Cost of sales:
   Products
41,485 
40,899 
155,084 
149,476 
   Services
5,589 
5,280 
16,411 
15,319 
Total cost of sales
47,074 
46,179 
171,495 
164,795 
Gross margin
35,885 
35,255 
132,687 
117,662 
Operating expenses:
Research and development
6,797 
5,717 
19,490 
16,142 
Selling, general and administrative
6,012 
5,412 
18,654 
16,357 
Total operating expenses
12,809 
11,129 
38,144 
32,499 
Operating income
23,076 
24,1

In [27]:
# negative/error tests
try:
    embedding_provider.embed_query("")
except Exception as exc:
    print(
        type(exc).__name__,
        ":",
        exc,
    )

ValueError : query must not be empty


In [28]:
try:
    vector_store.search(
        query_vector=[0.1, 0.2],
        top_k=5,
    )
except Exception as exc:
    print(
        type(exc).__name__,
        ":",
        exc,
    )

ValueError : Query embedding dimension mismatch: expected 3072, received 2


In [29]:
# Test 1: exact financial fact
query = "What were Apple's total net sales for the three months ended June 25, 2022?"
query_vector = embedding_provider.embed_query(
    query
)

results = vector_store.search(
    query_vector=query_vector,
    top_k=3,
)

for rank, result in enumerate(results, start=1):
    print("=" * 100)
    print("Rank:", rank)
    print("Score:", result["score"])
    print("Source:", result["source"])
    print("Page:", result["page"])
    print("Chunk ID:", result["chunk_id"])
    print()
    print(result["text"][:1000])

2026-09-19 23:35:48,432 | INFO | httpx2 | HTTP Request: POST https://qubinexa-dev-ai-resource.services.ai.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2026-09-19 23:35:48,434 | INFO | src.observability.tracing | Trace completed | step=azure_query_embedding | duration_seconds=1.348
2026-09-19 23:35:48,440 | INFO | src.providers.vector_store.azure_search | Executing vector search | index=rag-index | top_k=3
2026-09-19 23:35:50,648 | INFO | src.observability.tracing | Trace completed | step=azure_search_vector_query | duration_seconds=2.208
2026-09-19 23:35:50,649 | INFO | src.providers.vector_store.azure_search | Vector search completed | results=3


Rank: 1
Score: 0.7294477
Source: 2022 Q3 AAPL.pdf
Page: 4
Chunk ID: 2022_Q3_AAPL_p4_c1

PART I — FINANCIAL INFORMATION
Item 1.    Financial Statements
Apple Inc.
CONDENSED CONSOLIDATED STATEMENTS OF OPERATIONS (Unaudited)
(In millions, except number of shares which are reflected in thousands and per share amounts)
Three Months Ended
Nine Months Ended
June 25,
2022
June 26,
2021
June 25,
2022
June 26,
2021
Net sales:
   Products
$
63,355 
$
63,948 
$
245,241 
$
232,309 
   Services
19,604 
17,486 
58,941 
50,148 
Total net sales
82,959 
81,434 
304,182 
282,457 
Cost of sales:
   Products
41,485 
40,899 
155,084 
149,476 
   Services
5,589 
5,280 
16,411 
15,319 
Total cost of sales
47,074 
46,179 
171,495 
164,795 
Gross margin
35,885 
35,255 
132,687 
117,662 
Operating expenses:
Research and development
6,797 
5,717 
19,490 
16,142 
Selling, general and administrative
6,012 
5,412 
18,654 
16,357 
Total operating expenses
12,809 
11,129 
38,144 
32,499 
Operating income
23,076 
24,12

In [30]:
# Test 2: paraphrased query
query = "How much revenue did Apple generate during the quarter?"
query_vector = embedding_provider.embed_query(
    query
)

results = vector_store.search(
    query_vector=query_vector,
    top_k=3,
)

for rank, result in enumerate(results, start=1):
    print("=" * 100)
    print("Rank:", rank)
    print("Score:", result["score"])
    print("Source:", result["source"])
    print("Page:", result["page"])
    print("Chunk ID:", result["chunk_id"])
    print()
    print(result["text"][:1000])

2026-09-19 23:37:09,871 | INFO | httpx2 | HTTP Request: POST https://qubinexa-dev-ai-resource.services.ai.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2026-09-19 23:37:09,873 | INFO | src.observability.tracing | Trace completed | step=azure_query_embedding | duration_seconds=0.633
2026-09-19 23:37:09,880 | INFO | src.providers.vector_store.azure_search | Executing vector search | index=rag-index | top_k=3
2026-09-19 23:37:11,678 | INFO | src.observability.tracing | Trace completed | step=azure_search_vector_query | duration_seconds=1.797
2026-09-19 23:37:11,679 | INFO | src.providers.vector_store.azure_search | Vector search completed | results=3


Rank: 1
Score: 0.67380285
Source: 2022 Q3 AAPL.pdf
Page: 4
Chunk ID: 2022_Q3_AAPL_p4_c1

PART I — FINANCIAL INFORMATION
Item 1.    Financial Statements
Apple Inc.
CONDENSED CONSOLIDATED STATEMENTS OF OPERATIONS (Unaudited)
(In millions, except number of shares which are reflected in thousands and per share amounts)
Three Months Ended
Nine Months Ended
June 25,
2022
June 26,
2021
June 25,
2022
June 26,
2021
Net sales:
   Products
$
63,355 
$
63,948 
$
245,241 
$
232,309 
   Services
19,604 
17,486 
58,941 
50,148 
Total net sales
82,959 
81,434 
304,182 
282,457 
Cost of sales:
   Products
41,485 
40,899 
155,084 
149,476 
   Services
5,589 
5,280 
16,411 
15,319 
Total cost of sales
47,074 
46,179 
171,495 
164,795 
Gross margin
35,885 
35,255 
132,687 
117,662 
Operating expenses:
Research and development
6,797 
5,717 
19,490 
16,142 
Selling, general and administrative
6,012 
5,412 
18,654 
16,357 
Total operating expenses
12,809 
11,129 
38,144 
32,499 
Operating income
23,076 
24,1

In [31]:
# Test 3: another content type
query = "How many Apple shares were outstanding in July 2022?"
query_vector = embedding_provider.embed_query(
    query
)

results = vector_store.search(
    query_vector=query_vector,
    top_k=3,
)

for rank, result in enumerate(results, start=1):
    print("=" * 100)
    print("Rank:", rank)
    print("Score:", result["score"])
    print("Source:", result["source"])
    print("Page:", result["page"])
    print("Chunk ID:", result["chunk_id"])
    print()
    print(result["text"][:1000])

2026-09-19 23:38:22,100 | INFO | httpx2 | HTTP Request: POST https://qubinexa-dev-ai-resource.services.ai.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2026-09-19 23:38:22,103 | INFO | src.observability.tracing | Trace completed | step=azure_query_embedding | duration_seconds=0.625
2026-09-19 23:38:22,110 | INFO | src.providers.vector_store.azure_search | Executing vector search | index=rag-index | top_k=3
2026-09-19 23:38:23,531 | INFO | src.observability.tracing | Trace completed | step=azure_search_vector_query | duration_seconds=1.421
2026-09-19 23:38:23,533 | INFO | src.providers.vector_store.azure_search | Vector search completed | results=3


Rank: 1
Score: 0.6843771
Source: 2022 Q3 AAPL.pdf
Page: 2
Chunk ID: 2022_Q3_AAPL_p2_c1

If an emerging growth company, indicate by check mark if the Registrant has elected not to use the extended transition period for complying with any new or revised financial
accounting standards provided pursuant to Section 13(a) of the Exchange Act. ☐
Indicate by check mark whether the Registrant is a shell company (as defined in Rule 12b-2 of the Exchange Act).
Yes  ☐     No  ☒
16,070,752,000 shares of common stock were issued and outstanding as of July 15, 2022.
Rank: 2
Score: 0.679936
Source: 2022 Q3 AAPL.pdf
Page: 4
Chunk ID: 2022_Q3_AAPL_p4_c1

PART I — FINANCIAL INFORMATION
Item 1.    Financial Statements
Apple Inc.
CONDENSED CONSOLIDATED STATEMENTS OF OPERATIONS (Unaudited)
(In millions, except number of shares which are reflected in thousands and per share amounts)
Three Months Ended
Nine Months Ended
June 25,
2022
June 26,
2021
June 25,
2022
June 26,
2021
Net sales:
   Products
$
63,355 
$

In [32]:
query = "What was Apple's net income for the three months ended June 25, 2022?"
query_vector = embedding_provider.embed_query(
    query
)

results = vector_store.search(
    query_vector=query_vector,
    top_k=3,
)

for rank, result in enumerate(results, start=1):
    print("=" * 100)
    print("Rank:", rank)
    print("Score:", result["score"])
    print("Source:", result["source"])
    print("Page:", result["page"])
    print("Chunk ID:", result["chunk_id"])
    print()
    print(result["text"][:1000])

2026-09-19 23:39:28,913 | INFO | httpx2 | HTTP Request: POST https://qubinexa-dev-ai-resource.services.ai.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2026-09-19 23:39:28,915 | INFO | src.observability.tracing | Trace completed | step=azure_query_embedding | duration_seconds=0.636
2026-09-19 23:39:28,924 | INFO | src.providers.vector_store.azure_search | Executing vector search | index=rag-index | top_k=3
2026-09-19 23:39:30,339 | INFO | src.observability.tracing | Trace completed | step=azure_search_vector_query | duration_seconds=1.414
2026-09-19 23:39:30,339 | INFO | src.providers.vector_store.azure_search | Vector search completed | results=3


Rank: 1
Score: 0.7450473
Source: 2022 Q3 AAPL.pdf
Page: 5
Chunk ID: 2022_Q3_AAPL_p5_c1

Apple Inc.
CONDENSED CONSOLIDATED STATEMENTS OF COMPREHENSIVE INCOME (Unaudited)
(In millions)
Three Months Ended
Nine Months Ended
June 25,
2022
June 26,
2021
June 25,
2022
June 26,
2021
Net income
$
19,442 
$
21,744 
$
79,082 
$
74,129 
Other comprehensive income/(loss):
Change in foreign currency translation, net of tax
(721)
188 
(1,102)
659 
Change in unrealized gains/losses on derivative instruments, net of tax:
Change in fair value of derivative instruments
852 
(24)
1,548 
4 
Adjustment for net (gains)/losses realized and included in net income
121 
17 
(87)
593 
Total change in unrealized gains/losses on derivative instruments
973 
(7)
1,461 
597 
Change in unrealized gains/losses on marketable debt securities, net of tax:
Change in fair value of marketable debt securities
(3,150)
217 
(9,959)
(558)
Adjustment for net (gains)/losses realized and included in net income
95 
(54)
140 
(234)
To

In [ ]:
# tiny evaluation table

tests = [
    {
        "query": "What were Apple's total net sales for the three months ended June 25, 2022?",
        "expected_page": 4,
    },
    {
        "query": "How much revenue did Apple generate during the quarter?",
        "expected_page": 4,
    },
    {
        "query": "How many Apple shares were outstanding in July 2022?",
        "expected_page": 2,
    },
    {
        "query": "What was Apple's net income for the three months ended June 25, 2022?",
        "expected_page": 5,
    },
]

In [34]:
for test in tests:
    print("=" * 100)
    print("QUERY:", test["query"])

    query_vector = embedding_provider.embed_query(
        test["query"]
    )

    results = vector_store.search(
        query_vector=query_vector,
        top_k=3,
    )

    for rank, result in enumerate(results, start=1):
        print(
            f"Rank {rank} | "
            f"Page {result['page']} | "
            f"Score {result['score']}"
        )

    print(
        "Expected page:",
        test["expected_page"],
    )

QUERY: What were Apple's total net sales for the three months ended June 25, 2022?


2026-09-19 23:41:25,679 | INFO | httpx2 | HTTP Request: POST https://qubinexa-dev-ai-resource.services.ai.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2026-09-19 23:41:25,680 | INFO | src.observability.tracing | Trace completed | step=azure_query_embedding | duration_seconds=1.102
2026-09-19 23:41:25,687 | INFO | src.providers.vector_store.azure_search | Executing vector search | index=rag-index | top_k=3
2026-09-19 23:41:27,617 | INFO | src.observability.tracing | Trace completed | step=azure_search_vector_query | duration_seconds=1.929
2026-09-19 23:41:27,618 | INFO | src.providers.vector_store.azure_search | Vector search completed | results=3


Rank 1 | Page 4 | Score 0.7294477
Rank 2 | Page 5 | Score 0.7168779
Rank 3 | Page 3 | Score 0.68780756
Expected page: 4
QUERY: How much revenue did Apple generate during the quarter?


2026-09-19 23:41:28,114 | INFO | httpx2 | HTTP Request: POST https://qubinexa-dev-ai-resource.services.ai.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2026-09-19 23:41:28,116 | INFO | src.observability.tracing | Trace completed | step=azure_query_embedding | duration_seconds=0.498
2026-09-19 23:41:28,124 | INFO | src.providers.vector_store.azure_search | Executing vector search | index=rag-index | top_k=3
2026-09-19 23:41:29,430 | INFO | src.observability.tracing | Trace completed | step=azure_search_vector_query | duration_seconds=1.306
2026-09-19 23:41:29,431 | INFO | src.providers.vector_store.azure_search | Vector search completed | results=3


Rank 1 | Page 4 | Score 0.67380285
Rank 2 | Page 5 | Score 0.6661272
Rank 3 | Page 3 | Score 0.6496382
Expected page: 4
QUERY: How many Apple shares were outstanding in July 2022?


2026-09-19 23:41:29,814 | INFO | httpx2 | HTTP Request: POST https://qubinexa-dev-ai-resource.services.ai.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2026-09-19 23:41:29,837 | INFO | src.observability.tracing | Trace completed | step=azure_query_embedding | duration_seconds=0.404
2026-09-19 23:41:29,848 | INFO | src.providers.vector_store.azure_search | Executing vector search | index=rag-index | top_k=3
2026-09-19 23:41:31,148 | INFO | src.observability.tracing | Trace completed | step=azure_search_vector_query | duration_seconds=1.298
2026-09-19 23:41:31,154 | INFO | src.providers.vector_store.azure_search | Vector search completed | results=3


Rank 1 | Page 2 | Score 0.6843771
Rank 2 | Page 4 | Score 0.679936
Rank 3 | Page 5 | Score 0.6724883
Expected page: 2
QUERY: What was Apple's net income for the three months ended June 25, 2022?


2026-09-19 23:41:31,912 | INFO | httpx2 | HTTP Request: POST https://qubinexa-dev-ai-resource.services.ai.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2026-09-19 23:41:31,915 | INFO | src.observability.tracing | Trace completed | step=azure_query_embedding | duration_seconds=0.759
2026-09-19 23:41:31,925 | INFO | src.providers.vector_store.azure_search | Executing vector search | index=rag-index | top_k=3
2026-09-19 23:41:33,228 | INFO | src.observability.tracing | Trace completed | step=azure_search_vector_query | duration_seconds=1.301
2026-09-19 23:41:33,229 | INFO | src.providers.vector_store.azure_search | Vector search completed | results=3


Rank 1 | Page 5 | Score 0.7450473
Rank 2 | Page 4 | Score 0.74035805
Rank 3 | Page 3 | Score 0.69782317
Expected page: 5


In [35]:
def hit_at_k(
    results,
    expected_page,
):
    return any(
        result["page"] == expected_page
        for result in results
    )

In [36]:
correct = 0

for test in tests:

    query_vector = embedding_provider.embed_query(
        test["query"]
    )

    results = vector_store.search(
        query_vector=query_vector,
        top_k=3,
    )

    hit = hit_at_k(
        results,
        test["expected_page"],
    )

    print(
        test["query"],
        "->",
        "PASS" if hit else "FAIL",
    )

    correct += int(hit)

print(
    "Hit@3:",
    correct / len(tests),
)

2026-09-19 23:41:51,004 | INFO | httpx2 | HTTP Request: POST https://qubinexa-dev-ai-resource.services.ai.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2026-09-19 23:41:51,005 | INFO | src.observability.tracing | Trace completed | step=azure_query_embedding | duration_seconds=0.765
2026-09-19 23:41:51,013 | INFO | src.providers.vector_store.azure_search | Executing vector search | index=rag-index | top_k=3
2026-09-19 23:41:52,328 | INFO | src.observability.tracing | Trace completed | step=azure_search_vector_query | duration_seconds=1.314
2026-09-19 23:41:52,329 | INFO | src.providers.vector_store.azure_search | Vector search completed | results=3


What were Apple's total net sales for the three months ended June 25, 2022? -> PASS


2026-09-19 23:41:52,726 | INFO | httpx2 | HTTP Request: POST https://qubinexa-dev-ai-resource.services.ai.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2026-09-19 23:41:52,728 | INFO | src.observability.tracing | Trace completed | step=azure_query_embedding | duration_seconds=0.398
2026-09-19 23:41:52,738 | INFO | src.providers.vector_store.azure_search | Executing vector search | index=rag-index | top_k=3
2026-09-19 23:41:54,038 | INFO | src.observability.tracing | Trace completed | step=azure_search_vector_query | duration_seconds=1.300
2026-09-19 23:41:54,040 | INFO | src.providers.vector_store.azure_search | Vector search completed | results=3


How much revenue did Apple generate during the quarter? -> PASS


2026-09-19 23:41:54,915 | INFO | httpx2 | HTTP Request: POST https://qubinexa-dev-ai-resource.services.ai.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2026-09-19 23:41:54,916 | INFO | src.observability.tracing | Trace completed | step=azure_query_embedding | duration_seconds=0.876
2026-09-19 23:41:54,923 | INFO | src.providers.vector_store.azure_search | Executing vector search | index=rag-index | top_k=3
2026-09-19 23:41:56,225 | INFO | src.observability.tracing | Trace completed | step=azure_search_vector_query | duration_seconds=1.302
2026-09-19 23:41:56,226 | INFO | src.providers.vector_store.azure_search | Vector search completed | results=3


How many Apple shares were outstanding in July 2022? -> PASS


2026-09-19 23:41:56,728 | INFO | httpx2 | HTTP Request: POST https://qubinexa-dev-ai-resource.services.ai.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2026-09-19 23:41:56,729 | INFO | src.observability.tracing | Trace completed | step=azure_query_embedding | duration_seconds=0.502
2026-09-19 23:41:56,735 | INFO | src.providers.vector_store.azure_search | Executing vector search | index=rag-index | top_k=3
2026-09-19 23:41:58,034 | INFO | src.observability.tracing | Trace completed | step=azure_search_vector_query | duration_seconds=1.299
2026-09-19 23:41:58,035 | INFO | src.providers.vector_store.azure_search | Vector search completed | results=3


What was Apple's net income for the three months ended June 25, 2022? -> PASS
Hit@3: 1.0


In [37]:
query = "What risks did Apple identify about supply chain disruptions?"
query_vector = embedding_provider.embed_query(
    query
)

results = vector_store.search(
    query_vector=query_vector,
    top_k=3,
)

for rank, result in enumerate(results, start=1):
    print("=" * 100)
    print("Rank:", rank)
    print("Score:", result["score"])
    print("Source:", result["source"])
    print("Page:", result["page"])
    print("Chunk ID:", result["chunk_id"])
    print()
    print(result["text"][:1000])

2026-09-19 23:42:35,221 | INFO | httpx2 | HTTP Request: POST https://qubinexa-dev-ai-resource.services.ai.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2026-09-19 23:42:35,223 | INFO | src.observability.tracing | Trace completed | step=azure_query_embedding | duration_seconds=1.350
2026-09-19 23:42:35,229 | INFO | src.providers.vector_store.azure_search | Executing vector search | index=rag-index | top_k=3
2026-09-19 23:42:36,542 | INFO | src.observability.tracing | Trace completed | step=azure_search_vector_query | duration_seconds=1.312
2026-09-19 23:42:36,543 | INFO | src.providers.vector_store.azure_search | Vector search completed | results=3


Rank: 1
Score: 0.6395882
Source: 2022 Q3 AAPL.pdf
Page: 3
Chunk ID: 2022_Q3_AAPL_p3_c1

Apple Inc.
Form 10-Q
For the Fiscal Quarter Ended June 25, 2022
TABLE OF CONTENTS
Page
Part I
Item 1.
Financial Statements
1
Item 2.
Management’s Discussion and Analysis of Financial Condition and Results of Operations
14
Item 3.
Quantitative and Qualitative Disclosures About Market Risk
19
Item 4.
Controls and Procedures
19
Part II
Item 1.
Legal Proceedings
20
Item 1A.
Risk Factors
20
Item 2.
Unregistered Sales of Equity Securities and Use of Proceeds
20
Item 3.
Defaults Upon Senior Securities
21
Item 4.
Mine Safety Disclosures
21
Item 5.
Other Information
21
Item 6.
Exhibits
21
Rank: 2
Score: 0.6151821
Source: 2022 Q3 AAPL.pdf
Page: 4
Chunk ID: 2022_Q3_AAPL_p4_c1

PART I — FINANCIAL INFORMATION
Item 1.    Financial Statements
Apple Inc.
CONDENSED CONSOLIDATED STATEMENTS OF OPERATIONS (Unaudited)
(In millions, except number of shares which are reflected in thousands and per share amounts)
Three Mon

In [ ]:
# Test the service with only 10 chunks first
from src.services.indexing_service import (
    IndexingService,
)

In [39]:
indexing_service = IndexingService(
    embedding_provider=embedding_provider,
    vector_store=vector_store,
    batch_size=5,
)

In [40]:
summary = indexing_service.index_chunks(
    chunks[:10]
)

summary

2026-09-19 23:58:05,849 | INFO | src.services.indexing_service | Starting full corpus indexing | chunks=10 | batch_size=5
2026-09-19 23:58:05,850 | INFO | src.services.indexing_service | Processing indexing batch | batch=1 | range=1-5 | total=10
2026-09-19 23:58:07,710 | INFO | httpx2 | HTTP Request: POST https://qubinexa-dev-ai-resource.services.ai.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2026-09-19 23:58:07,814 | INFO | src.observability.tracing | Trace completed | step=azure_embedding_batch | duration_seconds=1.963
2026-09-19 23:58:07,814 | INFO | src.observability.tracing | Trace completed | step=indexing_batch_embedding | duration_seconds=1.964
2026-09-19 23:58:07,815 | INFO | src.providers.vector_store.azure_search | Uploading documents to Azure AI Search | count=5 | index=rag-index
2026-09-19 23:58:10,301 | INFO | src.observability.tracing | Trace completed | step=azure_search_document_upload | duration_seconds=2.486

{'total_chunks': 10, 'indexed_chunks': 10, 'batches': 2}

In [41]:
indexing_service = IndexingService(
    embedding_provider=embedding_provider,
    vector_store=vector_store,
    batch_size=50,
)

In [42]:
summary = indexing_service.index_chunks(
    chunks
)

summary

2026-09-19 23:59:26,258 | INFO | src.services.indexing_service | Starting full corpus indexing | chunks=1260 | batch_size=50
2026-09-19 23:59:26,266 | INFO | src.services.indexing_service | Processing indexing batch | batch=1 | range=1-50 | total=1260
2026-09-19 23:59:31,462 | INFO | httpx2 | HTTP Request: POST https://qubinexa-dev-ai-resource.services.ai.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2026-09-19 23:59:32,317 | INFO | src.observability.tracing | Trace completed | step=azure_embedding_batch | duration_seconds=6.051
2026-09-19 23:59:32,318 | INFO | src.observability.tracing | Trace completed | step=indexing_batch_embedding | duration_seconds=6.052
2026-09-19 23:59:32,319 | INFO | src.providers.vector_store.azure_search | Uploading documents to Azure AI Search | count=50 | index=rag-index
2026-09-19 23:59:36,192 | INFO | src.observability.tracing | Trace completed | step=azure_search_document_upload | duration_second

{'total_chunks': 1260, 'indexed_chunks': 1260, 'batches': 26}

In [43]:
queries = [
    "What were Apple's total net sales in Q3 2022?",
    "What was Microsoft's revenue performance?",
    "What did NVIDIA report about data center revenue?",
    "What risks did Intel discuss?",
]

In [45]:
for query in queries:

    print("=" * 100)
    print("QUERY:", query)

    query_vector = (
        embedding_provider.embed_query(
            query
        )
    )

    results = vector_store.search(
        query_vector=query_vector,
        top_k=5,
    )

    for rank, result in enumerate(
        results,
        start=1,
    ):
        print(
            f"\nRank {rank} | "
            f"Score={result['score']:.4f} | "
            f"Source={result['source']} | "
            f"Page={result['page']}"
        )

        print(
            result["text"][:500]
        )

QUERY: What were Apple's total net sales in Q3 2022?


2026-09-20 00:08:22,012 | INFO | httpx2 | HTTP Request: POST https://qubinexa-dev-ai-resource.services.ai.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2026-09-20 00:08:22,015 | INFO | src.observability.tracing | Trace completed | step=azure_query_embedding | duration_seconds=1.365
2026-09-20 00:08:22,023 | INFO | src.providers.vector_store.azure_search | Executing vector search | index=rag-index | top_k=5
2026-09-20 00:08:25,091 | INFO | src.observability.tracing | Trace completed | step=azure_search_vector_query | duration_seconds=3.067
2026-09-20 00:08:25,092 | INFO | src.providers.vector_store.azure_search | Vector search completed | results=5



Rank 1 | Score=0.7365 | Source=2023 Q3 AAPL.pdf | Page=19
Products and Services Performance
The following table shows net sales by category for the three- and nine-month periods ended July 1, 2023 and June 25, 2022 (dollars in millions):
Three Months Ended
Nine Months Ended
July 1,
2023
June 25,
2022
Change
July 1,
2023
June 25,
2022
Change
Net sales by category:
iPhone
$
39,669 
$
40,665 
(2)%
$
156,778 
$
162,863 
(4)%
Mac
6,840 
7,382 
(7)%
21,743 
28,669 
(24)%
iPad
5,791 
7,224 
(20)%
21,857 
22,118 
(1)%
Wearables, Home and Accessories
8,284 
8,0

Rank 2 | Score=0.7328 | Source=2022 Q3 AAPL.pdf | Page=18
COVID-19
The COVID-19 pandemic has had, and continues to have, a significant impact around the world, prompting governments and businesses to take unprecedented
measures, such as restrictions on travel and business operations, temporary closures of businesses, and quarantine and shelter-in-place orders. The COVID-19
pandemic has at times significantly curtailed global economic a

2026-09-20 00:08:25,670 | INFO | httpx2 | HTTP Request: POST https://qubinexa-dev-ai-resource.services.ai.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2026-09-20 00:08:25,671 | INFO | src.observability.tracing | Trace completed | step=azure_query_embedding | duration_seconds=0.578
2026-09-20 00:08:25,677 | INFO | src.providers.vector_store.azure_search | Executing vector search | index=rag-index | top_k=5
2026-09-20 00:08:27,118 | INFO | src.observability.tracing | Trace completed | step=azure_search_vector_query | duration_seconds=1.440
2026-09-20 00:08:27,118 | INFO | src.providers.vector_store.azure_search | Vector search completed | results=5



Rank 1 | Score=0.7249 | Source=2023 Q1 MSFT.pdf | Page=37
Revenue increased $1.1 billion or 7%.
 
•
Office Commercial products and cloud services revenue increased $627 million or 7%. Office 365 Commercial revenue grew
11% with seat growth of 12%, driven by small and medium business and frontline worker offerings, as well as growth in revenue
per user. Office Commercial products revenue declined 30% driven by continued customer shift to cloud offerings.
 
•
Office Consumer products and cloud services revenue decreased $39 million or 2%. Microsoft 365 C

Rank 2 | Score=0.7124 | Source=2023 Q3 MSFT.pdf | Page=27
PART I
Item 1
 
Segment revenue and operating income were as follows during the periods presented:
 
(In millions)
 
   
 
 
 
 
 
 
 
 
Three Months Ended September 30,
 
2023 
 
2022 
 
 
 
 
 
 
 
Revenue
 
 
 
 
Productivity and Business Processes
 $
18,592 $
16,465 
Intelligent Cloud
 
24,259 
20,325 
More Personal Computing
 
13,666 
13,332 
 
 
 
 
 
 
 
 
 
 
 
 
Total
 

2026-09-20 00:08:27,718 | INFO | httpx2 | HTTP Request: POST https://qubinexa-dev-ai-resource.services.ai.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2026-09-20 00:08:27,720 | INFO | src.observability.tracing | Trace completed | step=azure_query_embedding | duration_seconds=0.601
2026-09-20 00:08:27,727 | INFO | src.providers.vector_store.azure_search | Executing vector search | index=rag-index | top_k=5
2026-09-20 00:08:29,270 | INFO | src.observability.tracing | Trace completed | step=azure_search_vector_query | duration_seconds=1.542
2026-09-20 00:08:29,271 | INFO | src.providers.vector_store.azure_search | Vector search completed | results=5



Rank 1 | Score=0.7798 | Source=2023 Q3 NVDA.pdf | Page=28
services to deliver unique value. Our platforms address four large markets where our expertise is critical: Data Center, Gaming, Professional
Visualization, and Automotive.
Revenue for the third quarter of fiscal year 2024 was $18.12 billion, up 206% from a year ago and up 34% sequentially.
Data Center revenue was up 279% from a year ago and up 41% sequentially. Strong sales of the NVIDIA HGX platform were driven by global
demand for the training and inferencing of large language models, recomme

Rank 2 | Score=0.7716 | Source=2023 Q2 NVDA.pdf | Page=27
Second Quarter of Fiscal Year 2024 Summary
Three Months Ended
 
July 30, 2023
April 30, 2023
July 31, 2022
Quarter-over-Quarter
Change
Year-over-Year
Change
($ in millions, except per share data)
Revenue
$
13,507 
$
7,192 
$
6,704 
88 %
101 %
Gross margin
70.1 %
64.6 %
43.5 %
5.5 pts
26.6 pts
Operating expenses
$
2,662 
$
2,508 
$
2,416 
6 %
10 %
Operating income
$
6,800 
$
2,14

2026-09-20 00:08:29,841 | INFO | httpx2 | HTTP Request: POST https://qubinexa-dev-ai-resource.services.ai.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
2026-09-20 00:08:29,842 | INFO | src.observability.tracing | Trace completed | step=azure_query_embedding | duration_seconds=0.570
2026-09-20 00:08:29,849 | INFO | src.providers.vector_store.azure_search | Executing vector search | index=rag-index | top_k=5
2026-09-20 00:08:31,303 | INFO | src.observability.tracing | Trace completed | step=azure_search_vector_query | duration_seconds=1.453
2026-09-20 00:08:31,304 | INFO | src.providers.vector_store.azure_search | Vector search completed | results=5



Rank 1 | Score=0.7355 | Source=2023 Q3 INTC.pdf | Page=4
Table of Contents
•
cybersecurity and privacy risks;
•
investment and transaction risk;
•
IP risks and risks associated with litigation and regulatory proceedings;
•
evolving regulatory and legal requirements across many jurisdictions;
•
geopolitical and international trade conditions, including the impacts of Russia's war on Ukraine, recent events in Israel and rising tensions between the US and China;
•
our debt obligations and our ability to access sources of capital;
•
risks of large scale g

Rank 2 | Score=0.7341 | Source=2023 Q1 INTC.pdf | Page=4
•
cybersecurity and privacy risks;
•
investment and transaction risk;
•
IP risks and risks associated with litigation and regulatory proceedings;
•
evolving regulatory and legal requirements across many jurisdictions;
•
geopolitical and international trade conditions;
•
our debt obligations;
•
risks of large scale global operations;. 
•
macroeconomic conditions;
•
impacts of the C

| Query                         | Dense retrieval observation                                                |
| ----------------------------- | -------------------------------------------------------------------------- |
| Apple Q3 2022 net sales       | Relevant 2022 chunk retrieved, but a 2023 comparative filing ranked higher |
| Microsoft revenue performance | Relevant MSFT revenue discussions ranked highly                            |
| NVIDIA Data Center revenue    | Highly relevant NVDA Data Center passage ranked #1                         |
| Intel risks                   | Relevant Intel risk disclosures ranked #1–#3                               |
